In [ ]:
!head ratings_long.csv

userId,movieId,rating
0,16,5
0,72,5
0,86,5
0,259,1
0,319,4
0,521,4
0,534,2
0,671,1
0,673,2


In [ ]:
r = np.full((20, 1000),fill_value=np.nan)

In [61]:
df = pd.read_csv('ratings_long.csv')

In [63]:
for rec in df.itertuples():
    r[rec.userId][rec.movieId] = rec.rating

Note that $r$ matrix is $20 \times 1000$ with only <1\% full (highly sparse)

In [65]:
r

array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan,  4., nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]])

What if we define two matricies
- u = $20 \times 4$
- v = $4 \times 1000$


Then model $r$ as $u \times v$

Problem is we have to learn for $20 \times 4 + 4 \times 1000 = 4080$ parameters (better than 20.000 x 0.99 missing values)

## Problem

1. Define a convex loss function wrt $u$ and $v$
- Solve using gradient descent algorithm explained in **I Do**
- Use any regulatizer $L1$ or $L2$ to prevent overfitting

### Kayıp Fonksiyonu (Loss Function)

$r$ matrisini $u \times v$ çarpımıyla yaklaşık ifade etmeye çalışıyoruz. Ancak matris çok seyrek olduğundan **yalnızca gözlemlenen reytingler** üzerinden öğrenmeliyiz; boş girişler kayıp hesabına katılmamalı.

Bu nedenle kayıp fonksiyonunu şöyle tanımlıyoruz:

$$L = \underbrace{\sum_{(i,j) \in \Omega} (r_{ij} - u_i \cdot v_j)^2}_{\text{gözlemlenen hatanın karesi}} + \underbrace{\lambda \left(\|u\|_F^2 + \|v\|_F^2\right)}_{\text{L2 düzeltici}}$$

- $\Omega$ = gözlemlenen (NaN olmayan) reyting konumlarının kümesi  
- **İlk terim:** Tahmin hatalarının kareler toplamı — hem $u$ hem de $v$'ye göre **ayrı ayrı convex** olduğundan gradient descent ile optimize edilebilir  
- **İkinci terim:** L2 düzeltici — parametrelerin gereğinden büyümesini cezalandırır, böylece az gözlemlenen kullanıcı/film için **overfitting** önlenir; ders notasyonuyla `lamb` değişkeni ile ifade edilir

---

### Gradyanların Türetimi

Hata matrisini $E$ ile gösterelim; gözlemlenen konumlarda $E_{ij} = r_{ij} - \hat{r}_{ij}$, gözlemlenmeyenlerde $E_{ij} = 0$:

$$\frac{\partial L}{\partial u} = -2 \cdot E \cdot v^T + 2\lambda u \quad \Rightarrow \quad \texttt{g\_u}$$

$$\frac{\partial L}{\partial v} = -2 \cdot u^T \cdot E + 2\lambda v \quad \Rightarrow \quad \texttt{g\_v}$$

L2 türevi $2\lambda \cdot \text{parametre}$ şeklinde; bu derste anlatılan L2 gradyan formülüyle birebir örtüşmektedir.

---

### Gradient Descent Güncellemeleri

Ders notasyonuna uygun olarak sabit adım büyüklüğü `step` ile:

$$u \leftarrow u - \texttt{step} \cdot g_u, \qquad v \leftarrow v - \texttt{step} \cdot g_v$$

Yakınsama koşulu olarak $\|g_u\| + \|g_v\| < 0.01$ kontrol edilir; bu eşiğin altına inildiğinde döngü erken sonlandırılır.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

# r matrisini oluştur ve CSV'den doldur
r  = np.full((20, 1000), fill_value=np.nan)
df = pd.read_csv('ratings_long.csv')
for rec in df.itertuples():
    r[rec.userId][rec.movieId] = rec.rating

# Kayıp fonksiyonu: sadece gözlemlenen reytinglerdeki MSE
def loss_fn(error_obs):
    return (error_obs ** 2).sum()

np.random.seed(0)
k    = 4       # latent faktör sayısı
lamb = 0.01    # L2 düzeltici katsayısı (regularizer)
step = 0.001   # adım büyüklüğü
n_epoch = 3000

u = np.random.randn(20, k)    * 0.1   # 20 × 4
v = np.random.randn(k, 1000)  * 0.1   # 4 × 1000

mask   = ~np.isnan(r)   # True → gözlemlenen reyting
losses = []

for i in range(n_epoch):
    r_hat = u @ v
    # Gözlemlenmemiş girişleri sıfırla, gradyana katkıda bulunmasınlar
    error = np.where(mask, r - r_hat, 0.0)   # 20 × 1000

    loss = loss_fn(error[mask]) + lamb * ((u ** 2).sum() + (v ** 2).sum())
    losses.append(loss)

    # Analitik gradyanlar (L2 regularizer türevi: 2·lamb·parametre)
    g_u = -2 * error @ v.T + 2 * lamb * u   # 20 × 4
    g_v = -2 * u.T @ error + 2 * lamb * v   # 4 × 1000

    if np.linalg.norm(g_u) + np.linalg.norm(g_v) < 0.01:
        print(f"Epoch {i}'de yakınsadı")
        break

    u -= step * g_u
    v -= step * g_v

    if i % 500 == 0:
        rmse = np.sqrt(((r_hat[mask] - r[mask]) ** 2).mean())
        print(f"Epoch {i:5d}: loss={loss:.4f}  RMSE={rmse:.4f}")

r_hat = u @ v
final_rmse = np.sqrt(((r_hat[mask] - r[mask]) ** 2).mean())
print(f"\nSon eğitim RMSE: {final_rmse:.4f}")

### Kayıp Eğrisi Yorumu

Aşağıdaki grafik, her epoch'ta hesaplanan toplam kaybı göstermektedir. Kaybın monoton azalması, seçilen adım büyüklüğünün (`step = 0.001`) uygun olduğunu ve gradient descent'in düzgün çalıştığını doğrular. Eğri düzleştikçe model yakınsamıştır; L2 düzeltici sayesinde parametreler sonsuz büyümez ve kayıp kararlı bir platoya oturur.

In [ ]:
px.line(y=losses,
        title="Gradient Descent — Kayıp Eğrisi",
        labels={"index": "Epoch", "y": "Loss"}).show()

### Film Önerisi — Modelin Kullanımı

Eğitim tamamlandığında $\hat{r} = u \times v$ matrisi, **tüm kullanıcı-film çiftleri** için reyting tahmini içerir; hem gözlemlenmiş hem de gözlemlenmemiş pozisyonlar için. Matrix factorization'ın temel amacı da budur: seyrek bir matristen öğrenip eksik değerleri doldurmak.

Bir kullanıcıya öneri yapmak için:
1. O kullanıcının henüz izlemediği filmler seçilir (`mask` ile belirlenir)  
2. Bu filmler için $\hat{r}$ matrisinden tahmini reytingler okunur  
3. En yüksek tahmini reyting alan filmler önerilir

Bu, Netflix ve Spotify gibi sistemlerin temel algoritmasıdır.

In [ ]:
# Kullanıcı 0 için henüz izlemediği filmlerden en iyi 10 öneri
user_id = 0
not_seen = np.where(~mask[user_id])[0]        # izlenmemiş film indeksleri
preds    = r_hat[user_id, not_seen]            # o filmler için tahmini reyting
top10    = np.argsort(preds)[-10:][::-1]       # en yüksek 10 tahmin

print(f"Kullanıcı {user_id} için önerilen 10 film:")
for movie_id, pred in zip(not_seen[top10], preds[top10]):
    print(f"  Film {movie_id:4d}: tahmini reyting = {pred:.2f}")